# Module 4: Architecture & Wiring
### BWSI SGAI 2026

By the end of this module you'll know:
- How `main.py` is the single entry point for every game mode
- How `_make_components()` wires together the classifier, recommender, data parser, and scorekeeper
- How `GameEnv` (in `base_env.py`) sequences each game turn — classify, recommend, act
- Why `probs` are passed *into* `recommend()` instead of computed inside it
- How `TrainEnv` wraps the game loop with a Gymnasium interface for RL training

---
*Open this notebook in JupyterLab. Press `Alt+R` to enter slideshow mode.*  
*Students: just scroll through normally and run the exercise cells.*

## How Does One Turn Actually Work?

You press Save. Something happens. But how?

A single turn in `auto` mode touches **five files**:

```
main.py              ← entry point, mode selection, component wiring
game_interfaces/
  base_env.py        ← sequences classify → recommend → act
  data_parser.py     ← picks the next humanoid from the CSV
models/
  CNNClassifier.py   ← classifies the humanoid image
  RLRecommender.py   ← picks the action
```

This module traces the full wiring — from the command you type to the action that gets applied.  
No ML yet — that's Modules 5 and 6. This module is about the plumbing.

## Where We're Starting: `main.py`

```
bwsi-sgai/
├── main.py              ← now
├── gameplay/
├── game_interfaces/
├── models/
├── ui/
├── training/
└── tests/
```

`main.py` is the single entry point — every mode, every flag, every component combination routes through it.  
If you want to change *what* runs, you start here.

*`bwsi-sgai/main.py`*

## main.py — Locate

Open `main.py` now. It's the first file the Python interpreter runs. Structure at a glance:

```
main.py
├── _seed_all(seed)              — seeds Python, NumPy, PyTorch for reproducibility
├── _make_components(mode, ...)  — factory: returns classifier + recommender pair
├── class Main
│   ├── __init__(mode, log, ...)  — wires everything, launches game
│   └── _run_auto(...)           — headless RL loop (no GUI)
└── build_parser()               — defines the CLI flags
```

Two things to focus on: **`build_parser()`** (what flags exist) and **`_make_components()`** (where you'd swap models).

*`bwsi-sgai/main.py`*

## main.py — Orient: the four flags

```python
def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        prog="python main.py",
        description="SGAI game — run with -m to choose a mode",
    )
    parser.add_argument(
        "-m", "--mode",
        choices=["human", "machine", "human_cnn", "human_rl", "human_cnn_rl", "auto"],
        default="human",
    )
    parser.add_argument("-l", "--log",   action="store_true")  # write log.csv
    parser.add_argument("-d", "--data",  type=str)              # data directory
    parser.add_argument("-s", "--seed",  type=int, default=None)# reproducibility
    # ... plus a 'train' subcommand ...
    return parser
```

Four flags: **mode**, **log**, **data**, **seed**.  
The subcommand `train` routes to the training scripts — that's Module 7.

*`bwsi-sgai/main.py`*

## main.py — Touch: read the CLI help

`build_parser()` is extracted from `__main__` so we can import it without launching the game.  
Run the cell below to read the full help text from inside the notebook.

In [1]:
import os
import sys
from pathlib import Path
%matplotlib inline

# Ensure the project root is the working directory and is on sys.path,
# so local packages (game_interfaces, gameplay, models) are importable
# and relative paths like DATA_DIR="data" resolve correctly.
# Works whether Jupyter was launched from bwsi-sgai/ or lectures/.
_here = Path().resolve()
_root = _here.parent if _here.name == "lectures" else _here
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [2]:
from main import build_parser

print(build_parser().format_help())
# can also run from CLI with: python -m main --help

usage: python main.py [-h]
                      [-m {human,machine,human_cnn,human_rl,human_cnn_rl,auto}]
                      [-l] [-d DATA] [-s SEED]
                      {train} ...

SGAI game — run with -m to choose a mode

positional arguments:
  {train}
    train               Train a model (RL or CNN). This is a utility, not a
                        game mode.

options:
  -h, --help            show this help message and exit
  -m, --mode {human,machine,human_cnn,human_rl,human_cnn_rl,auto}
                        Game mode (default: human). human=pure human play;
                        machine=RL plays step-by-step with Advance Turn
                        button; human_cnn=CNN classifies, human decides;
                        human_rl=RL recommends action, human decides (CNN runs
                        internally, label hidden); human_cnn_rl=CNN+RL both
                        assist, human decides; auto=fully headless RL play (no
                        GUI)
  -l, --log

*`bwsi-sgai/main.py`*

## main.py — Orient: `_make_components()`

```python
def _make_components(
    mode: str,
    data_dir: str,
) -> tuple[HumanoidClassifier | None, ActionRecommender | None, bool, bool]:
    """Factory — returns (classifier, recommender, auto_execute, show_classifier_label)."""

    if mode == "human":
        return None, None, False, False

    if mode == "machine":
        return CNNClassifier(img_data_root=data_dir), RLRecommender(), True, True

    if mode == "human_cnn":
        return CNNClassifier(img_data_root=data_dir), None, False, True

    if mode == "human_rl":
        return CNNClassifier(img_data_root=data_dir), RLRecommender(), False, False

    if mode == "human_cnn_rl":
        return CNNClassifier(img_data_root=data_dir), RLRecommender(), False, True
    # ... ValueError for unknown modes
```

Each mode returns a different **combination** of components — no game logic changes, just different wiring.

*`bwsi-sgai/main.py`*

## The Interesting Seam: why assemble in one place?

**Why does `_make_components()` exist at all?**  
It could inline `CNNClassifier(...)` and `RLRecommender()` directly in `Main.__init__()`.  
Instead, every mode routes through this single factory.

The reason: **swappability**.

If you want to test `OracleClassifier` instead of `CNNClassifier`, you change exactly one line in one function:

```python
# Before:
return CNNClassifier(img_data_root=data_dir), RLRecommender(), True, True

# After (swap the classifier):
return OracleClassifier(), RLRecommender(), True, True
```

The rest of the codebase — `GameEnv`, `UI`, `_run_auto` — never knows the difference.  
They all speak the `HumanoidClassifier` interface. That's the point of the abstraction.

## Now: inside `game_interfaces/`

```
bwsi-sgai/
├── main.py
├── gameplay/
├── game_interfaces/        ← now
│   ├── base_env.py        ← now (the game loop)
│   ├── train_env.py       ← now (RL training wrapper)
│   ├── data_parser.py     ← brief mention (full coverage: Module 5)
│   └── obs_builder.py
├── models/
├── ui/
├── training/
└── tests/
```

`game_interfaces/` is the layer that connects game rules to data, the UI, and RL training.  
`base_env.py` is the heart — it sequences every turn.

*`bwsi-sgai/game_interfaces/base_env.py`*

## base_env.py — Locate

Open `game_interfaces/base_env.py`. This is the shared base class for the RL training environment:

```
game_interfaces/base_env.py
└── class GameEnv(Env)                 ← Gymnasium Env base class
    ├── __init__(root, w, h,
    │           data_parser,            ← injected: supplies humanoids
    │           scorekeeper,            ← injected: tracks time, score
    │           img_data_root,
    │           display)
    ├── observation_space               ← Box([0,1], shape=(15,))
    ├── action_space                    ← Discrete(4)
    ├── reset()                         ← zeros obs, resets scorekeeper + data_parser
    └── step()  [abstract]              ← must be implemented by subclasses
```

`GameEnv` does **not** import `CNNClassifier` or `RLRecommender` directly.  
It takes `data_parser` and `scorekeeper` as constructor parameters — the classifier is handled by the subclass.

*`bwsi-sgai/game_interfaces/base_env.py`*

## base_env.py — Orient: observation and action spaces

```python
class GameEnv(Env):
    """Observation space — 15 floats, all in [0, 1]:

    Index  Name                  Encoding
    0      time_bin              5 bins (0=critical ... 4=fresh), normalized / 4
    1-4    class_counts          per-class ambulance occupancy, normalized / capacity
    5-8    humanoid_class        one-hot over 4 classes (zombie, healthy, injured, corpse)
    9      humanoid_confidence   CNN confidence bin (0=low, 1=med, 2=high), normalized / 2
    10-13  doable_actions        binary availability of each action
    14     humanoid_value_bin    5-bin value score, normalized / 4
    """

    self.observation_space = spaces.Box(
        low=0.0, high=1.0, shape=(self.obs_size,), dtype=np.float32
    )
    self.action_space = spaces.Discrete(self.scorekeeper.num_actions)  # 4 actions
```

The observation is a **flat float32 vector of 15 numbers** — everything the RL agent knows about the current state, encoded as values between 0 and 1.  
`OBS_SIZE = 15` is defined in `gameplay/constants.py`.

*`bwsi-sgai/game_interfaces/base_env.py`*

## base_env.py — Orient: dependency injection

```python
def __init__(
    self,
    root: Any,
    w: int,
    h: int,
    data_parser: DataParser,      # ← passed in, not created here
    scorekeeper: ScoreKeeper,     # ← passed in, not created here
    img_data_root: str = "data",
    display: bool = False,
) -> None:
    self.data_parser = data_parser
    self.scorekeeper = scorekeeper
    # ... other setup ...
```

**`GameEnv` doesn't build its own `DataParser` or `ScoreKeeper`** — it receives them as arguments.  
This is *dependency injection*: the caller decides what objects to provide.  

The benefit: you can swap in a different `DataParser` (say, one with a fixed seed) without touching `GameEnv` at all.

*`bwsi-sgai/game_interfaces/base_env.py`*

## The Interesting Seam: why does `recommend()` take `probs`?

Look at how a turn runs in `_run_auto()` (in `main.py`) and `TrainEnv.step()` (in `train_env.py`):

```python
# Step 1: classify the humanoid image
probs = classifier.classify(humanoid)

# Step 2: recommend an action — pass probs in, don't call classify again
action = recommender.recommend(humanoid, probs, scorekeeper)
```

**Why not have `recommend()` call `classifier.classify()` itself?**

Two reasons:

1. **Single forward pass per turn** — running the CNN is expensive. Classify once, use the result twice (once for the label the player sees, once to build the RL observation vector).

2. **Decoupling** — `RLRecommender` never needs to know *which* classifier was used. It just receives 4 probabilities. Swap `CNNClassifier` for `OracleClassifier` and `RLRecommender` doesn't change at all.

The probs flow: `classifier.classify()` → `probs` → into `recommend()` AND into the observation vector.

*`bwsi-sgai/game_interfaces/base_env.py`*

## base_env.py — Touch: inspect the spaces

`GameEnv` is abstract — we can't instantiate it directly.  
But we can inspect the constants that define its spaces without building the full environment.

In [3]:
from gameplay.constants import OBS_SIZE
from gameplay.scorekeeper import ScoreKeeper
from gameplay.constants import SHIFT_LENGTH, AMBULANCE_CAPACITY

sk = ScoreKeeper(shift_length=SHIFT_LENGTH, capacity=AMBULANCE_CAPACITY, seed=42)

print(f"Observation vector length: {OBS_SIZE} slots")
print(f"Number of actions:         {sk.num_actions}")
print()
print("Observation slot layout:")
slots = [
    ("[0]",     "time_bin",             "remaining-time bin (0=critical, 4=fresh)"),
    ("[1-4]",   "class_counts",         "ambulance occupancy per class"),
    ("[5-8]",   "humanoid_class",       "one-hot CNN prediction"),
    ("[9]",     "humanoid_confidence",  "CNN confidence bin"),
    ("[10-13]", "doable_actions",       "which actions are currently legal"),
    ("[14]",    "humanoid_value_bin",   "triage priority bin"),
]
for idx, name, desc in slots:
    print(f"  {idx:<8} {name:<25} {desc}")

Observation vector length: 15 slots
Number of actions:         4

Observation slot layout:
  [0]      time_bin                  remaining-time bin (0=critical, 4=fresh)
  [1-4]    class_counts              ambulance occupancy per class
  [5-8]    humanoid_class            one-hot CNN prediction
  [9]      humanoid_confidence       CNN confidence bin
  [10-13]  doable_actions            which actions are currently legal
  [14]     humanoid_value_bin        triage priority bin


INSTRUCTOR NOTES:
Ask students: which slots change every turn? (humanoid_class, humanoid_confidence, humanoid_value_bin, doable_actions)
Which slots change less often? (time_bin — only as time ticks; class_counts — only after save/scram)
The RL agent uses all 15 to decide what to do. It learned — from training — which slots matter most.
Students can look at obs_builder.py to see exactly how each slot is computed.

## Now: `train_env.py` — the RL wrapper

```
bwsi-sgai/
├── main.py
├── gameplay/
├── game_interfaces/
│   ├── base_env.py        (done)
│   ├── train_env.py       ← now
│   ├── data_parser.py     ← brief mention
│   └── obs_builder.py
├── models/
├── ui/
├── training/
└── tests/
```

`TrainEnv` extends `GameEnv` with the full Gymnasium interface the DQN training loop expects.  
This is where the game becomes a proper RL environment.

*`bwsi-sgai/game_interfaces/train_env.py`*

## train_env.py — Locate

Open `game_interfaces/train_env.py`. It's a subclass of `GameEnv`:

```
game_interfaces/train_env.py
└── class TrainEnv(GameEnv)
    ├── __init__(..., classifier, ...)  ← takes a classifier as a parameter
    ├── reset()                          ← zeros obs, loads first humanoid
    ├── get_humanoid()                   ← draws from DataParser + classifies
    ├── _build_obs()                     ← fills the 15-slot observation vector
    └── step(action_idx)                 ← execute action, compute reward, advance turn
```

`TrainEnv` adds the classifier and the Gymnasium `step()` implementation.  
`GameEnv` provides `observation_space`, `action_space`, and `reset()` infrastructure.

*`bwsi-sgai/game_interfaces/train_env.py`*

## train_env.py — Orient: `step()`

```python
def step(self, action_idx: int) -> tuple[np.ndarray, float, bool, bool, dict]:
    action = list(ActionState)[action_idx]            # 0=save, 1=squish, 2=skip, 3=scram

    action_executed = self.scorekeeper.map_do_action(action, self.humanoid)

    if action_executed:
        current_cum_reward = self.scorekeeper.get_cumulative_reward()
        reward = current_cum_reward - self.previous_cum_reward  # delta reward
        self.previous_cum_reward = current_cum_reward
    else:
        reward = BLOCKED_ACTION_PENALTY   # -0.5: don't try illegal actions

    if self.scorekeeper.remaining_time <= 0 or len(self.data_parser.unvisited) == 0:
        episode_done = True
    else:
        self.get_humanoid()   # draw next humanoid + classify it

    self._build_obs()         # update obs vector for next step
    return self.obs.copy(), reward, episode_done, False, {}
```

Reward is computed as a **delta** — the change in cumulative reward since last turn, not an absolute value.  
`BLOCKED_ACTION_PENALTY = -0.5` teaches the agent not to try actions it can't take (e.g., Save when ambulance is full).

*`bwsi-sgai/game_interfaces/train_env.py`*

## train_env.py — Orient: `reset()` and the Gymnasium contract

The four things a Gymnasium environment must provide:

| What | Where in `TrainEnv` |
|------|--------------------|
| `observation_space` | Inherited from `GameEnv` — `Box([0,1], shape=(15,))` |
| `action_space` | Inherited from `GameEnv` — `Discrete(4)` |
| `reset()` | Overrides `GameEnv.reset()` — zeros obs, resets scorekeeper + data_parser, loads first humanoid |
| `step(action_idx)` | Defined in `TrainEnv` — execute action, compute reward, advance turn |

```python
def reset(self, *, seed=None, options=None) -> tuple[np.ndarray, dict]:
    super().reset(seed=seed, options=options)   # zeros obs, resets scorekeeper + data_parser
    self._episode_done = False
    self.get_humanoid()    # draw first humanoid + classify it
    self._build_obs()      # populate obs slots
    return self.obs.copy(), {}
```

The training loop calls `reset()` at the start of each episode and `step()` for every turn.  
That's the whole Gymnasium contract.

*`bwsi-sgai/game_interfaces/train_env.py` · `bwsi-sgai/game_interfaces/data_parser.py`*

## A Brief Note: DataParser

Inside `TrainEnv.get_humanoid()`:

```python
def get_humanoid(self) -> None:
    self.humanoid = self.data_parser.get_random()          # draw from CSV
    self.humanoid_probs = self.classifier.classify(self.humanoid)  # classify it
```

`DataParser` is the other key dependency — it reads `data/consolidated_metadata.csv` and draws humanoids one at a time without replacement.

```
DataParser
├── unvisited: set[int]   — row indices not yet drawn
├── get_random()          — draw one humanoid, remove from unvisited
└── reset()               — restore all rows for a new episode
```

`DataParser` is covered fully in **Module 5**.  
For now: it's the source of `Humanoid` objects, and `TrainEnv` uses it without needing to know the CSV details.

*`bwsi-sgai/game_interfaces/train_env.py`*

## train_env.py — Touch: inspect the spaces

`TrainEnv` requires a classifier at construction time, so we can't instantiate it without model weights.  
But we can inspect the space definitions directly from the Gymnasium module — no weights needed.

In [4]:
from gymnasium import spaces
import numpy as np
from gameplay.constants import OBS_SIZE
from gameplay.scorekeeper import ScoreKeeper
from gameplay.constants import SHIFT_LENGTH, AMBULANCE_CAPACITY

sk = ScoreKeeper(shift_length=SHIFT_LENGTH, capacity=AMBULANCE_CAPACITY, seed=42)

# Reproduce the exact spaces TrainEnv creates — without instantiating the env
observation_space = spaces.Box(low=0.0, high=1.0, shape=(OBS_SIZE,), dtype=np.float32)
action_space = spaces.Discrete(sk.num_actions)

print(f"observation_space: {observation_space}")
print(f"  shape:  {observation_space.shape}")
print(f"  dtype:  {observation_space.dtype}")
print(f"  low:    {observation_space.low[0]},  high: {observation_space.high[0]}")
print()
print(f"action_space:      {action_space}")
print(f"  n: {action_space.n} actions (0=save, 1=squish, 2=skip, 3=scram)")

observation_space: Box(0.0, 1.0, (15,), float32)
  shape:  (15,)
  dtype:  float32
  low:    0.0,  high: 1.0

action_space:      Discrete(4)
  n: 4 actions (0=save, 1=squish, 2=skip, 3=scram)


INSTRUCTOR NOTES:
Point out that action_space.n=4 matches the 4 entries in ActionState enum.
Ask: what would happen to the saved weights if you added a 5th action? (OBS_SIZE and action_space both need to change; old weights won't load — you must retrain.)
The observation_space Box uses float32 — same type the DQN network expects as input.
Students curious about the network architecture: that's Module 6 (DQN.py, DQNAgent.py).

*`bwsi-sgai/main.py` · `bwsi-sgai/game_interfaces/base_env.py`*

## Exercise: Trace a Full Turn

This module's exercise is different: **no value to change — just trace the execution path.**

Here is what happens during turn 1 of `python main.py -m auto --seed 42`:

```
Step 1  python main.py -m auto --seed 42
        └── Main.__init__(mode="auto", ...)
               └── _seed_all(42)
               └── DataParser(data_dir, seed=42)
               └── ScoreKeeper(SHIFT_LENGTH, AMBULANCE_CAPACITY, seed=42)
               └── _run_auto(data_parser, scorekeeper, log)

Step 2  _run_auto() sets up components
        └── CNNClassifier(img_data_root=data_dir)   ← classifies images
        └── RLRecommender()                         ← picks actions

Step 3  while loop — turn 1:
        └── humanoid = data_parser.get_random()     ← one row from CSV
        └── probs = classifier.classify(humanoid)   ← 4 probabilities from CNN
        └── action = recommender.recommend(humanoid, probs, scorekeeper)
        └── scorekeeper.map_do_action(action, humanoid)

Step 4  Loop continues until scorekeeper.remaining_time <= 0
        or data_parser.unvisited is empty
```

*`bwsi-sgai/main.py` · `bwsi-sgai/game_interfaces/base_env.py`*

## Exercise: Verify the Injection Seam

Run the cell below. It reproduces the component-wiring step from `_run_auto()` **without** launching the game:  
assemble the components, run one manual turn, print what happened.

**No weights needed** — `OracleClassifier` reads the ground-truth state directly from the `Humanoid`, so it always returns perfect probs.

In [5]:
import numpy as np
from game_interfaces.data_parser import DataParser
from gameplay.scorekeeper import ScoreKeeper
from gameplay.constants import SHIFT_LENGTH, AMBULANCE_CAPACITY
from models.OracleClassifier import OracleClassifier
from models.HeuristicRecommender import HeuristicRecommender
from models.HumanoidClassifier import probs_to_class

# --- Assemble components (same pattern as _make_components / _run_auto) ---
data_parser = DataParser("data", seed=42)
scorekeeper = ScoreKeeper(SHIFT_LENGTH, AMBULANCE_CAPACITY, seed=42)

# Using OracleClassifier + HeuristicRecommender: no trained weights required
classifier = OracleClassifier()
recommender = HeuristicRecommender()

# --- Simulate turn 1 ---
humanoid = data_parser.get_random()           # draw from CSV
probs = classifier.classify(humanoid)         # 4 probabilities
action = recommender.recommend(humanoid, probs, scorekeeper)  # pick action
scorekeeper.map_do_action(action, humanoid)   # apply action

label, confidence = probs_to_class(probs)

print(f"Humanoid state (ground truth): {humanoid.state}")
print(f"Classifier predicted:          {label.value}  (confidence: {confidence:.3f})")
print(f"Recommender chose:             {action.value}")
print(f"Time remaining after turn 1:   {scorekeeper.remaining_time} min")
print(f"Unvisited humanoids remaining: {len(data_parser.unvisited)}")

Humanoid state (ground truth): healthy
Classifier predicted:          healthy  (confidence: 1.000)
Recommender chose:             save
Time remaining after turn 1:   690 min
Unvisited humanoids remaining: 199


*`bwsi-sgai/main.py` · `bwsi-sgai/game_interfaces/base_env.py`*

## Exercise: Try Swapping a Component

**The swap — 5 minutes:**

In the cell above, change line:
```python
classifier = OracleClassifier()
```
to:
```python
from models.CNNClassifier import CNNClassifier
classifier = CNNClassifier(img_data_root="data")
```

Re-run the cell. **Everything else stays the same.** The recommender, scorekeeper, and data parser never know you changed the classifier.

**Questions to think about:**
- Does `recommender.recommend(humanoid, probs, scorekeeper)` change at all? Why not?
- Does the action change? If so, why?
- Where in the codebase would you look to understand *why* the CNN predicted that label?

---
*If you don't have trained CNN weights (`models/weights/baselineCNN.pth`), `CNNClassifier` will raise a `FileNotFoundError`.  
Stick with `OracleClassifier` for now — Module 5 covers training.*

INSTRUCTOR NOTES:
Walk through the turn-tracing table on the board before students run the cell.
Key question: what does OracleClassifier do differently from CNNClassifier? (It reads humanoid.state directly — no image processing. Perfect accuracy always.)
When students swap in CNNClassifier: the output may differ from OracleClassifier because the CNN can be wrong. That's the point — the architecture handles both cases identically.
Students with trained weights: encourage them to run a few turns in a loop (copy the while-loop pattern from _run_auto in main.py) and compare actions between Oracle and CNN.
Students without weights: that's fine — OracleClassifier is a legitimate diagnostic tool, not a workaround.

## Pop Quiz — Navigation

These are navigation questions, not recall. Which file do you open?

---

**Q1.** You want to swap in a different classifier — replace `CNNClassifier` with `OracleClassifier`.  
Which function in which file do you edit?

A) `game_interfaces/base_env.py` — `GameEnv.__init__()`  
B) `main.py` — `_make_components()`  
C) `models/CNNClassifier.py` — change the class name  
D) `game_interfaces/train_env.py` — `TrainEnv.__init__()`

**Answer: B — `main.py`, `_make_components()`**  
That function is the single factory for all component combinations.  
Change the classifier there and every mode that uses it picks up the swap automatically.

## Pop Quiz — Navigation

**Q2.** The RL agent uses a 15-slot observation vector each turn.  
Which class **builds** that vector by calling `classify()` and filling in the slots?

A) `game_interfaces/base_env.py` — `GameEnv`  
B) `main.py` — `Main`  
C) `game_interfaces/train_env.py` — `TrainEnv`  
D) `models/RLRecommender.py` — `RLRecommender`

**Answer: C — `game_interfaces/train_env.py`, `TrainEnv`**  
`TrainEnv._build_obs()` calls `build_obs()` from `obs_builder.py` to fill the 15-slot vector.  
`GameEnv` defines the *structure* of the space; `TrainEnv` does the actual population each turn.  
`RLRecommender` receives the finished obs vector — it never builds it.

## Pop Quiz — Navigation

**Q3.** Why does `recommender.recommend(humanoid, probs, scorekeeper)` take `probs` as a parameter  
instead of calling `classifier.classify(humanoid)` itself inside `recommend()`?

A) Because `recommend()` is an abstract method and can't call other methods  
B) To avoid a circular import between `models/` and `game_interfaces/`  
C) To run the CNN only once per turn and keep the recommender decoupled from the classifier  
D) Because `probs` is computed by `DataParser`, not the classifier

**Answer: C — single forward pass, and decoupling**  
Running the CNN is the expensive step — classifying once and passing `probs` into `recommend()` avoids running it twice.  
And `RLRecommender` never needs to know *which* classifier produced those probs.  
Swap `CNNClassifier` for `OracleClassifier` in `_make_components()` and `RLRecommender.recommend()` never changes.

## What You Now Know

- `main.py` — entry point; `build_parser()` defines flags; `_make_components()` is where you swap classifiers or recommenders
- `game_interfaces/base_env.py` — `GameEnv` owns the obs/action space definitions and `reset()` infrastructure; `step()` is abstract
- `game_interfaces/train_env.py` — `TrainEnv` implements `step()` and `_build_obs()`; the full Gymnasium contract lives here
- `probs` flow: `classifier.classify()` → `probs` → into `recommend()` AND into the obs vector — one pass, two uses
- Dependency injection: `GameEnv` and `TrainEnv` don't build their own components — they receive them, making swaps cheap

The architecture in one line:  
**`main.py` wires it. `base_env.py` sequences it. `train_env.py` exposes it to the RL loop.**

---

**Next: Module 5 — Perception (CNN + DataParser)**  
How does the CNN classify a humanoid image? Where does the data come from?  
We'll open `HumanoidClassifier.py`, `CNNClassifier.py`, `OracleClassifier.py`, and `data_parser.py`.